In [4]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# Panama boundary
panama = (
    ee.FeatureCollection('FAO/GAUL/2015/level0')
    .filter(ee.Filter.eq('ADM0_NAME', 'Panama'))
)

# Dynamic World dataset

# (Alternative) Panama boundary
# countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")

# panama = countries.filter(
#     ee.Filter.eq('country_na', 'Panama')
# ).geometry()

# Date range
START = '2024-01-01'
END = '2024-12-31'

# Dynamic World collection
dw_col = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filterBounds(panama)
    .filterDate(START, END)
)

# Sentinel-2 collection
s2_col = (
    ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
    .filterBounds(panama)
    .filterDate(START, END)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

print("DW images:", dw_col.size().getInfo())
print("S2 images:", s2_col.size().getInfo())

# Create mosaics/composites
# Dynamic World label mode composite
dw_mode = dw_col.select('label').mode()

# Sentinel-2 median composite
s2_median = s2_col.median()

# Visualization palettes
VIS_PALETTE = [
    '419bdf',  # water
    '397d49',  # trees
    '88b053',  # grass
    '7a87c6',  # flooded vegetation
    'e49635',  # crops
    'dfc35a',  # shrub/scrub
    'c4281b',  # built
    'a59b8f',  # bare
    'b39fe1',  # snow/ice
]

# Visualize Dynamic World
dw_vis = dw_mode.visualize(
    min=0,
    max=8,
    palette=VIS_PALETTE
)

# Map
Map = geemap.Map()

Map.centerObject(panama, 7)

# Sentinel-2 RGB
Map.addLayer(
    s2_median,
    {
        'bands': ['B4', 'B3', 'B2'],
        'min': 0,
        'max': 3000
    },
    'Sentinel-2 Median'
)

# Dynamic World
Map.addLayer(
    dw_vis,
    {},
    'Dynamic World Land Cover'
)

# Panama boundary
Map.addLayer(
    panama,
    {'color': 'red'},
    'Panama Boundary'
)

Map.addLayerControl()

# MERIT Topography

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")

panama = countries.filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

roi = panama.geometry()

# Sentinel-2 cloud mask
def mask_s2_clouds(image):

    qa = image.select("QA60")

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    return image.updateMask(mask).divide(10000)

# Sentinel-2 collection
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate("2024-01-01", "2025-05-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
    .map(mask_s2_clouds)
)

# Composite clipped to Panama
image = s2.median().clip(roi)

# Visualization parameters
rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 0.3,
}

# Load DEM and clip to Panama
dem = ee.Image("MERIT/DEM/v1_0_3").clip(roi)

# Elevation visualization
dem_vis = {
    "min": 0,
    "max": 3500,
    "palette": [
        "#000000",
        "#478fcd",
        "#86c58e",
        "#afc35e",
        "#8f7131",
        "#b78d4f",
        "#e2b8a6",
        "#ffffff"   # high mountains
    ],
}

# Add layers
Map.addLayer(image, rgb_vis, "Sentinel-2")

Map.addLayer(dem, dem_vis, "Elevation")

# Center map
Map.centerObject(roi, 10)

Map

DW images: 838
S2 images: 540


Map(center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position', 'transparent_bg…